In [3]:
 !apt update && apt install -y ffmpeg
 !pip install yt-dlp Pillow requests ffmpeg-python

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [99.9 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,049 kB]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,258 kB]
Get:

In [4]:
# ============================================================
# THE TRUE LENS — INGESTOR AGENT (Full Version)
# ============================================================
# Handles ALL input types:
#  - Local image / video / audio files
#  - YouTube links
#  - Facebook / Instagram / TikTok / Twitter links
#  - Direct media URLs (.mp4, .jpg, etc.)
#  - HLS streams (.m3u8)
#
# PIPELINE:
# Input → Classify → Sanitize → Download → Detect Type
#       → Normalize → Fingerprint → JSON Output
#
# INSTALL FIRST (run in Colab/Kaggle):

# ============================================================

import os
import json
import logging
import hashlib
import mimetypes
import tempfile
import subprocess
import requests
from pathlib import Path
from datetime import datetime
from urllib.parse import urlparse

import yt_dlp
from PIL import Image

logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s] [INGESTOR] %(message)s",
    datefmt="%H:%M:%S"
)
log = logging.getLogger(__name__)


# ============================================================
# MODULE 1: INPUT CLASSIFIER
# ============================================================

def classify_input(user_input: str) -> str:
    log.info(f"Classifying input: {user_input}")

    if os.path.exists(user_input):
        return "local_file"

    try:
        parsed = urlparse(user_input)
        is_url = parsed.scheme in ("http", "https")
    except:
        return "unsupported"

    if not is_url:
        return "unsupported"

    url_lower = user_input.lower()

    if any(x in url_lower for x in ["youtube.com", "youtu.be"]):
        return "youtube_url"

    if any(x in url_lower for x in ["facebook.com", "fb.watch", "instagram.com",
                                      "tiktok.com", "twitter.com", "x.com",
                                      "whatsapp.com"]):
        return "social_media"

    if url_lower.endswith(".m3u8"):
        return "hls_stream"

    media_extensions = [".mp4", ".mov", ".avi", ".webm", ".mkv",
                        ".jpg", ".jpeg", ".png", ".webp", ".gif",
                        ".mp3", ".wav"]
    if any(url_lower.endswith(ext) for ext in media_extensions):
        return "direct_media"

    # Default — try with yt-dlp anyway (supports 1000+ sites)
    return "social_media"


# ============================================================
# MODULE 2: SECURITY SANITIZATION
# ============================================================

DANGEROUS_EXTENSIONS = [".exe", ".bat", ".sh", ".php", ".js", ".py",
                         ".dll", ".cmd", ".vbs", ".ps1"]

MAX_FILE_SIZE_MB = 500

def sanitize_input(user_input: str, input_type: str) -> bool:
    log.info("Running security checks...")

    if input_type == "local_file":
        path = Path(user_input)

        if path.suffix.lower() in DANGEROUS_EXTENSIONS:
            raise ValueError(f"BLOCKED: Dangerous file type '{path.suffix}'")

        size_mb = os.path.getsize(user_input) / (1024 * 1024)
        if size_mb > MAX_FILE_SIZE_MB:
            raise ValueError(f"BLOCKED: File too large ({size_mb:.1f}MB). Max is {MAX_FILE_SIZE_MB}MB")

        log.info(f"Local file passed security checks ({size_mb:.1f}MB)")

    else:
        parsed = urlparse(user_input)

        if parsed.scheme not in ("http", "https"):
            raise ValueError(f"BLOCKED: Unsafe URL scheme '{parsed.scheme}'")

        blocked_hosts = ["localhost", "127.0.0.1", "0.0.0.0", "169.254."]
        if any(b in parsed.netloc for b in blocked_hosts):
            raise ValueError("BLOCKED: Local network URLs are not allowed")

        log.info("URL passed security checks")

    return True


# ============================================================
# MODULE 3: MEDIA DOWNLOADER / EXTRACTOR
# ============================================================

def download_media(user_input: str, input_type: str, temp_dir: str, cookies_file: str = None) -> str:
    """
    cookies_file: optional path to a cookies.txt file for private/login-gated
    Facebook/Instagram content. Leave as None for public content.
    """

    if input_type == "local_file":
        log.info("Local file — skipping download step")
        return user_input

    if input_type in ("youtube_url", "social_media", "hls_stream"):
        log.info(f"Downloading via yt-dlp from: {user_input}")

        output_template = os.path.join(temp_dir, "%(title)s.%(ext)s")
        options = {
            "outtmpl": output_template,
            "format": "mp4[height<=480]/best[height<=480]/best",
            "quiet": True,
            "no_warnings": True,
        }

        # ── For private/login-gated FB or Instagram posts ──
        if cookies_file and os.path.exists(cookies_file):
            options["cookiefile"] = cookies_file
            log.info("Using cookies file for authenticated access")

        with yt_dlp.YoutubeDL(options) as ydl:
            info = ydl.extract_info(user_input, download=True)
            downloaded_path = ydl.prepare_filename(info)

        log.info(f"Downloaded to: {downloaded_path}")
        return downloaded_path

    if input_type == "direct_media":
        log.info(f"Downloading direct media from: {user_input}")

        response = requests.get(user_input, stream=True, timeout=30)
        response.raise_for_status()

        ext = Path(urlparse(user_input).path).suffix or ".tmp"
        output_path = os.path.join(temp_dir, f"downloaded_media{ext}")

        with open(output_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)

        log.info(f"Downloaded to: {output_path}")
        return output_path

    raise ValueError(f"Cannot download input type: {input_type}")


# ============================================================
# MODULE 4: MEDIA TYPE DETECTOR
# ============================================================

def detect_media_type(file_path: str) -> str:
    log.info(f"Detecting media type of: {file_path}")

    mime_type, _ = mimetypes.guess_type(file_path)

    if mime_type:
        if mime_type.startswith("video"):
            return "video"
        elif mime_type.startswith("image"):
            return "image"
        elif mime_type.startswith("audio"):
            return "audio"

    ext = Path(file_path).suffix.lower()
    video_exts = [".mp4", ".mov", ".avi", ".webm", ".mkv", ".flv"]
    image_exts = [".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif"]
    audio_exts = [".mp3", ".wav", ".aac", ".ogg", ".flac"]

    if ext in video_exts:
        return "video"
    elif ext in image_exts:
        return "image"
    elif ext in audio_exts:
        return "audio"

    return "unsupported"


# ============================================================
# MODULE 5: CANONICALIZATION ENGINE
# ============================================================

def canonicalize_video(input_path: str, output_path: str) -> str:
    log.info("Canonicalizing video to MP4 (H.264 + AAC)...")

    result = subprocess.run([
        "ffmpeg",
        "-i", input_path,
        "-c:v", "libx264",
        "-preset", "fast",
        "-c:a", "aac",
        "-movflags", "+faststart",
        "-y",
        output_path
    ], capture_output=True, text=True)

    if result.returncode != 0:
        raise RuntimeError(f"FFmpeg failed: {result.stderr}")

    log.info(f"Video saved to: {output_path}")
    return output_path


def canonicalize_image(input_path: str, output_path: str) -> str:
    log.info("Canonicalizing image to PNG...")

    img = Image.open(input_path)
    img = img.convert("RGB")
    img.save(output_path, "PNG")

    log.info(f"Image saved to: {output_path}")
    return output_path


def canonicalize_audio(input_path: str, output_path: str) -> str:
    log.info("Canonicalizing audio to WAV...")

    result = subprocess.run([
        "ffmpeg",
        "-i", input_path,
        "-ar", "44100",
        "-ac", "2",
        "-y",
        output_path
    ], capture_output=True, text=True)

    if result.returncode != 0:
        raise RuntimeError(f"FFmpeg failed: {result.stderr}")

    log.info(f"Audio saved to: {output_path}")
    return output_path


def canonicalize(file_path: str, media_type: str, output_dir: str) -> tuple:
    if media_type == "video":
        out = os.path.join(output_dir, "canonical_video.mp4")
        return canonicalize_video(file_path, out), "mp4"

    elif media_type == "image":
        out = os.path.join(output_dir, "canonical_image.png")
        return canonicalize_image(file_path, out), "png"

    elif media_type == "audio":
        out = os.path.join(output_dir, "canonical_audio.wav")
        return canonicalize_audio(file_path, out), "wav"

    else:
        raise ValueError(f"Unsupported media type: {media_type}")


# ============================================================
# MODULE 6: FINGERPRINT GENERATOR
# ============================================================

def generate_fingerprint(file_path: str) -> str:
    log.info("Generating SHA-256 fingerprint...")

    sha256 = hashlib.sha256()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            sha256.update(chunk)

    file_hash = sha256.hexdigest()
    log.info(f"Fingerprint: {file_hash}")
    return file_hash


# ============================================================
# MAIN PIPELINE — RUN INGESTOR AGENT
# ============================================================

def run_ingestor_agent(user_input: str, cookies_file: str = None) -> dict:
    """
    user_input: file path OR any URL (YouTube, Facebook, Instagram,
                TikTok, direct image/video link, HLS stream)
    cookies_file: optional, only needed for private/login-gated posts
    """
    log.info("=" * 50)
    log.info("INGESTOR AGENT STARTING")
    log.info("=" * 50)

    with tempfile.TemporaryDirectory() as temp_dir:
        try:
            input_type = classify_input(user_input)
            log.info(f"Input classified as: {input_type}")

            if input_type == "unsupported":
                raise ValueError("Input type not supported")

            sanitize_input(user_input, input_type)

            downloaded_path = download_media(user_input, input_type, temp_dir, cookies_file)

            media_type = detect_media_type(downloaded_path)
            log.info(f"Media type detected: {media_type}")

            if media_type == "unsupported":
                raise ValueError("Unsupported media type — cannot process")

            output_dir = os.getcwd()

            canonical_path, canonical_format = canonicalize(
                downloaded_path,
                media_type,
                output_dir
            )

            file_hash = generate_fingerprint(canonical_path)

            result = {
                "status": "success",
                "processed_at": datetime.now().isoformat(),
                "input_type": input_type,
                "media_type": media_type,
                "canonical_format": canonical_format,
                "file_hash": file_hash,
                "normalized_file_path": canonical_path,
                "ready_for_next_agent": True
            }

            log.info("INGESTOR AGENT COMPLETE ✓")
            log.info("Result: " + json.dumps(result, indent=2))
            return result

        except Exception as e:
            log.error(f"Ingestor Agent FAILED: {str(e)}")
            return {
                "status": "error",
                "error_message": str(e),
                "processed_at": datetime.now().isoformat(),
                "ready_for_next_agent": False
            }


# ============================================================
# TEST IT — change these to whatever you want to check
# ============================================================

if __name__ == "__main__":

    test_inputs = [
        # Local image
        # "/content/photo.jpg",

        # Local video
        "/content/brakobama.mp4",

        # YouTube link
        # "https://www.youtube.com/watch?v=aqz-KE-bpKQ",

        # Facebook post link (public only, unless you pass cookies_file)
        # "https://www.facebook.com/watch/?v=1234567890",

        # Direct image URL
        # "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png",
    ]

    for test_input in test_inputs:
        print(f"\n>>> Testing: {test_input}")
        result = run_ingestor_agent(test_input)
        print(f">>> Result status: {result['status']}\n")
        print(json.dumps(result, indent=2))


>>> Testing: /content/brakobama.mp4
>>> Result status: success

{
  "status": "success",
  "processed_at": "2026-06-30T13:46:55.590377",
  "input_type": "local_file",
  "media_type": "video",
  "canonical_format": "mp4",
  "file_hash": "c954f16f0feddbb4d44182f7afd773bdddca29c141bb31b838d2a97bbe3629c6",
  "normalized_file_path": "/content/canonical_video.mp4",
  "ready_for_next_agent": true
}


In [6]:
# ============================================================
# THE TRUE LENS — PROVENANCE AGENT (Agent 2)
# ============================================================
# PURPOSE: Analyze the digital "passport" of the media file.
# It examines metadata, checks C2PA signatures, calculates
# hashes, and gives a rapid authenticity pre-assessment.
#
# INPUT:  The JSON dict output from the Ingestor Agent
# OUTPUT: Structured JSON with provenance findings
#
# INSTALL (run this cell first in Colab/Kaggle):
# !pip install Pillow exifread mutagen hachoir requests
# ============================================================

# ── CHANGE THIS ──────────────────────────────────────────────
# Paste the full JSON output from Agent 1 here.
# Either paste it directly, or point to a saved .json file.
# ─────────────────────────────────────────────────────────────

import os
import json
import hashlib
import logging
import requests
from datetime import datetime
from pathlib import Path

# For metadata extraction
import exifread          # EXIF data from images
import mutagen           # Audio/video metadata
from PIL import Image
from PIL.ExifTags import TAGS

# ── LOGGING SETUP ──
logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s] [PROVENANCE] %(message)s",
    datefmt="%H:%M:%S"
)
log = logging.getLogger(__name__)


# ============================================================
# MODULE 1: METADATA EXTRACTOR
# Pulls all hidden data embedded inside the file
# ============================================================

def extract_image_metadata(file_path: str) -> dict:
    """
    Extracts EXIF metadata from images.
    EXIF contains: camera model, GPS, timestamp, software used, etc.
    AI-generated images typically have NO EXIF or suspicious software tags.
    """
    log.info("Extracting image EXIF metadata...")
    metadata = {}
    flags = []

    try:
        # Method 1: PIL EXIF extraction
        img = Image.open(file_path)
        exif_data = img._getexif()

        if exif_data:
            for tag_id, value in exif_data.items():
                tag_name = TAGS.get(tag_id, tag_id)
                metadata[str(tag_name)] = str(value)

            # ── FLAG: Check for AI/generative software signatures ──
            software = metadata.get("Software", "").lower()
            ai_keywords = ["midjourney", "stable diffusion", "dall-e", "firefly",
                           "runway", "synthesia", "deepfake", "faceswap", "reface",
                           "generated", "ai", "artificial"]
            for keyword in ai_keywords:
                if keyword in software:
                    flags.append(f"AI_SOFTWARE_DETECTED: Software tag says '{metadata.get('Software')}'")

            # ── FLAG: Missing camera make/model (real photos have this) ──
            if "Make" not in metadata and "Model" not in metadata:
                flags.append("NO_CAMERA_INFO: No camera make/model found — may be synthetic")

            # ── FLAG: Timestamp inconsistency ──
            if "DateTime" in metadata and "DateTimeOriginal" in metadata:
                if metadata["DateTime"] != metadata["DateTimeOriginal"]:
                    flags.append("TIMESTAMP_MISMATCH: File modified after creation")

        else:
            flags.append("NO_EXIF: Image has no EXIF data — suspicious for a 'real' photo")
            log.info("No EXIF data found in image")

    except Exception as e:
        log.warning(f"PIL EXIF extraction failed: {e}")
        flags.append(f"METADATA_READ_ERROR: {str(e)}")

    # Method 2: exifread for more detail
    try:
        with open(file_path, "rb") as f:
            tags = exifread.process_file(f, details=False)
            for key, val in tags.items():
                if key not in metadata:
                    metadata[key] = str(val)
    except Exception as e:
        log.warning(f"exifread extraction failed: {e}")

    return {"raw_metadata": metadata, "flags": flags}


def extract_video_metadata(file_path: str) -> dict:
    """
    Extracts metadata from video/audio files using mutagen.
    Looks for: encoder software, creation time, GPS data, comments.
    """
    log.info("Extracting video/audio metadata...")
    metadata = {}
    flags = []

    try:
        media_file = mutagen.File(file_path, easy=True)

        if media_file and media_file.tags:
            for key, val in media_file.tags.items():
                metadata[key] = str(val)

            # ── FLAG: Check encoder for AI tools ──
            encoder = str(metadata.get("encoder", "")).lower()
            comment = str(metadata.get("comment", "")).lower()

            ai_keywords = ["synthesia", "heygen", "deepfake", "did.com",
                           "runway", "sora", "generated", "artificial"]
            for keyword in ai_keywords:
                if keyword in encoder or keyword in comment:
                    flags.append(f"AI_ENCODER_DETECTED: Encoder/comment tag suggests AI generation")

        else:
            flags.append("NO_VIDEO_METADATA: No metadata tags found in video/audio file")
            log.info("No metadata tags found in media file")

    except Exception as e:
        log.warning(f"Mutagen metadata extraction failed: {e}")
        flags.append(f"METADATA_READ_ERROR: {str(e)}")

    return {"raw_metadata": metadata, "flags": flags}


# ============================================================
# MODULE 2: METADATA CONSISTENCY ANALYZER
# Looks for internal contradictions in the metadata
# ============================================================

def analyze_metadata_consistency(metadata: dict, media_type: str) -> list:
    """
    Checks for logical inconsistencies that reveal tampering.
    Returns a list of red flags found.
    """
    log.info("Analyzing metadata consistency...")
    flags = []
    raw = metadata.get("raw_metadata", {})

    if media_type == "image":
        # Check if GPS data exists but seems implausible
        if "GPS GPSLatitude" in raw and "GPS GPSLongitude" in raw:
            log.info("GPS data present in image — noted")
        else:
            # Not a flag by itself, just log it
            log.info("No GPS data in image (common for shared images)")

        # Check if color space is unusual for a real camera
        color_space = raw.get("ColorSpace", "")
        if color_space and "uncalibrated" in str(color_space).lower():
            flags.append("UNCALIBRATED_COLOR_SPACE: Unusual color space — possible synthetic origin")

    # Check for extremely round/suspicious file sizes
    # (AI generated content is often saved at standard sizes)

    return flags


# ============================================================
# MODULE 3: C2PA SIGNATURE CHECKER
# Checks for the industry-standard content authenticity signature
# ============================================================

def check_c2pa_signature(file_path: str) -> dict:
    """
    Checks whether the file has a valid C2PA (Content Credentials) signature.

    C2PA is the gold standard for content provenance — used by Adobe, BBC,
    Microsoft, Sony, and major news organizations.

    NOTE: Full C2PA verification requires the 'c2pa-python' library which
    needs Rust compilation. This module does a header-level check and
    attempts verification if the library is available.

    Returns a dict with c2pa_status and details.
    """
    log.info("Checking for C2PA digital signature...")

    result = {
        "c2pa_present": False,
        "c2pa_valid": False,
        "c2pa_issuer": None,
        "c2pa_timestamp": None,
        "c2pa_claims": [],
        "note": ""
    }

    # ── Method 1: Check for C2PA markers in raw bytes ──
    # C2PA embeds a 'jumbf' or 'c2pa' marker in the file binary
    try:
        with open(file_path, "rb") as f:
            header = f.read(65536)  # Read first 64KB to find markers

        c2pa_markers = [b"c2pa", b"jumbf", b"contentcredentials", b"cai.adobe.com"]
        found_markers = []

        for marker in c2pa_markers:
            if marker in header.lower():
                found_markers.append(marker.decode("utf-8", errors="ignore"))

        if found_markers:
            result["c2pa_present"] = True
            result["note"] = f"C2PA markers found: {found_markers}. Full verification requires c2pa-python library."
            log.info(f"C2PA markers detected: {found_markers}")
        else:
            result["note"] = "No C2PA signature detected in file"
            log.info("No C2PA signature found")

    except Exception as e:
        result["note"] = f"C2PA check failed: {str(e)}"
        log.warning(f"C2PA binary check failed: {e}")

    # ── Method 2: Try c2pa-python library if installed ──
    try:
        import c2pa
        reader = c2pa.Reader.from_file(file_path)
        manifest = reader.get_active_manifest()

        if manifest:
            result["c2pa_present"] = True
            result["c2pa_valid"] = True
            result["c2pa_issuer"] = manifest.get("claim_generator", "Unknown")
            result["c2pa_timestamp"] = manifest.get("timestamp", "")
            result["c2pa_claims"] = list(manifest.get("assertions", {}).keys())
            result["note"] = "C2PA signature VERIFIED via c2pa-python"
            log.info("C2PA signature verified successfully")

    except ImportError:
        log.info("c2pa-python not installed — using binary marker detection only")
    except Exception as e:
        result["note"] += f" | c2pa-python error: {str(e)}"
        log.warning(f"c2pa-python verification failed: {e}")

    return result


# ============================================================
# MODULE 4: HASH CROSS-REFERENCE
# Checks if this exact file has been seen before
# ============================================================

def cross_reference_hash(file_hash: str) -> dict:
    """
    Cross-references the file hash against known databases.

    In production, this connects to:
    - InVID/WeVerify hash database
    - Hive Moderation API
    - NCMEC PhotoDNA (for harmful content)
    - Your own internal database of verified/debunked content

    For now, this shows the structure and checks against a
    small demo lookup table.
    """
    log.info(f"Cross-referencing hash: {file_hash[:16]}...")

    # ── DEMO LOOKUP TABLE ──
    # In production, replace this with actual API calls
    # Example: requests.post("https://api.hivemoderation.com/...", json={"hash": file_hash})
    KNOWN_HASHES = {
        # Format: "hash": {"verdict": "...", "source": "...", "detail": "..."}
        "abc123demo": {
            "verdict": "DEBUNKED",
            "source": "Snopes",
            "detail": "This video was debunked in 2023 — original footage is from 2019"
        },
        "def456demo": {
            "verdict": "VERIFIED_AUTHENTIC",
            "source": "Reuters",
            "detail": "Verified by Reuters fact-checking team"
        },
    }

    if file_hash in KNOWN_HASHES:
        match = KNOWN_HASHES[file_hash]
        log.info(f"Hash match found: {match['verdict']}")
        return {
            "hash_match": True,
            "verdict": match["verdict"],
            "source": match["source"],
            "detail": match["detail"]
        }

    # ── REAL API CALL TEMPLATE ──
    # Uncomment and fill in your API key to use Hive Moderation:
    #
    # try:
    #     response = requests.post(
    #         "https://api.thehive.ai/api/v2/task/sync",
    #         headers={"token": "YOUR_HIVE_API_KEY"},
    #         json={"hash": file_hash}
    #     )
    #     data = response.json()
    #     if data.get("status") == "found":
    #         return {"hash_match": True, "verdict": data["verdict"], ...}
    # except Exception as e:
    #     log.warning(f"Hive API call failed: {e}")

    log.info("No known hash match — file not seen before")
    return {
        "hash_match": False,
        "verdict": "UNKNOWN",
        "source": None,
        "detail": "File not found in any known database — first-time submission"
    }


# ============================================================
# MODULE 5: PROVENANCE SCORE CALCULATOR
# Turns all findings into a single provenance score
# ============================================================

def calculate_provenance_score(
    metadata_flags: list,
    c2pa_result: dict,
    hash_result: dict
) -> tuple:
    """
    Calculates a provenance sub-score from 0-100.
    This feeds into the final Trust Score in the Synthesis step.

    Scoring logic:
    - Start at 50 (neutral — we don't know yet)
    - C2PA valid = +40 (strong positive signal)
    - C2PA present but unverified = +15
    - Hash match VERIFIED = +30
    - Hash match DEBUNKED = -60
    - Each metadata flag = -10 (up to -30 max)
    - No metadata at all = -15
    """
    score = 50
    reasoning = []

    # C2PA scoring
    if c2pa_result["c2pa_valid"]:
        score += 40
        reasoning.append("C2PA_VALID: Cryptographic signature verified — strong authenticity signal")
    elif c2pa_result["c2pa_present"]:
        score += 15
        reasoning.append("C2PA_PRESENT: C2PA markers found but full verification pending")
    else:
        score -= 5
        reasoning.append("NO_C2PA: No content credentials found — cannot verify origin")

    # Hash scoring
    if hash_result["hash_match"]:
        if hash_result["verdict"] == "VERIFIED_AUTHENTIC":
            score += 30
            reasoning.append(f"HASH_VERIFIED: Known authentic — source: {hash_result['source']}")
        elif hash_result["verdict"] == "DEBUNKED":
            score -= 60
            reasoning.append(f"HASH_DEBUNKED: Known fake — source: {hash_result['source']}")
    else:
        reasoning.append("HASH_UNKNOWN: First-time submission — no prior record")

    # Metadata flag scoring (cap penalty at 30)
    penalty = min(len(metadata_flags) * 10, 30)
    score -= penalty
    for flag in metadata_flags:
        reasoning.append(f"METADATA_FLAG: {flag}")

    # Clamp score to 0-100
    score = max(0, min(100, score))

    return score, reasoning


# ============================================================
# MAIN PIPELINE — RUN PROVENANCE AGENT
# ============================================================

def run_provenance_agent(ingestor_output: dict) -> dict:
    """
    MASTER FUNCTION — runs the full Provenance Agent pipeline.

    Input:  ingestor_output = the JSON dict from Agent 1
    Output: structured JSON with provenance findings for Agent 3 & 4
    """
    log.info("=" * 50)
    log.info("PROVENANCE AGENT STARTING")
    log.info("=" * 50)

    # ── Guard: only proceed if Ingestor succeeded ──
    if not ingestor_output.get("ready_for_next_agent"):
        return {
            "status": "error",
            "error_message": "Ingestor Agent did not complete successfully",
            "ready_for_next_agent": False
        }

    file_path = ingestor_output["normalized_file_path"]
    media_type = ingestor_output["media_type"]
    file_hash = ingestor_output["file_hash"]

    try:
        all_metadata_flags = []

        # ── STEP 1: Extract metadata ──
        if media_type == "image":
            meta_result = extract_image_metadata(file_path)
        else:
            meta_result = extract_video_metadata(file_path)

        raw_metadata = meta_result["raw_metadata"]
        all_metadata_flags.extend(meta_result["flags"])

        # ── STEP 2: Analyze metadata consistency ──
        consistency_flags = analyze_metadata_consistency(meta_result, media_type)
        all_metadata_flags.extend(consistency_flags)

        # ── STEP 3: C2PA check ──
        c2pa_result = check_c2pa_signature(file_path)

        # ── STEP 4: Hash cross-reference ──
        hash_result = cross_reference_hash(file_hash)

        # ── STEP 5: Calculate provenance score ──
        provenance_score, reasoning = calculate_provenance_score(
            all_metadata_flags, c2pa_result, hash_result
        )

        # ── STEP 6: Build output ──
        result = {
            "status": "success",
            "processed_at": datetime.now().isoformat(),
            "agent": "provenance",
            "file_path": file_path,
            "file_hash": file_hash,
            "metadata": {
                "raw": raw_metadata,
                "flags": all_metadata_flags
            },
            "c2pa": c2pa_result,
            "hash_lookup": hash_result,
            "provenance_score": provenance_score,
            "provenance_reasoning": reasoning,
            "chain_of_custody_established": (
                c2pa_result["c2pa_valid"] or
                hash_result.get("verdict") == "VERIFIED_AUTHENTIC"
            ),
            "ready_for_next_agent": True
        }

        log.info(f"PROVENANCE AGENT COMPLETE ✓ — Score: {provenance_score}/100")
        log.info("Result:\n" + json.dumps(
            {k: v for k, v in result.items() if k != "metadata"}, indent=2
        ))
        return result

    except Exception as e:
        log.error(f"Provenance Agent FAILED: {str(e)}")
        return {
            "status": "error",
            "error_message": str(e),
            "processed_at": datetime.now().isoformat(),
            "ready_for_next_agent": False
        }


# ============================================================
# ── HOW TO CHANGE FOR YOUR OWN TEST ──
#
# 1. Run Agent 1 first and get its JSON output
# 2. Paste that JSON output into the variable below
# 3. Run this cell
# ============================================================

if __name__ == "__main__":

    # ── PASTE AGENT 1 OUTPUT HERE ──
    # Copy the full JSON from Agent 1's output and paste it here:

    agent1_output = {
        "status": "success",
        "processed_at": "2026-05-13T07:37:01.844058",
        "input_type": "local_file",
        "media_type": "video",            # Change to "image" or "audio" as needed
        "canonical_format": "mp4",
        "file_hash": "c954f16f0feddbb4d44182f7afd773bdddca29c141bb31b838d2a97bbe3629c6",
        "normalized_file_path": "/content/canonical_video.mp4",  # ← CHANGE THIS to your actual file path
        "ready_for_next_agent": True
    }

    print("\n" + "="*60)
    print("  RUNNING PROVENANCE AGENT")
    print("="*60)

    result = run_provenance_agent(agent1_output)

    print("\n>>> FINAL PROVENANCE RESULT:")
    print(json.dumps(result, indent=2))


  RUNNING PROVENANCE AGENT

>>> FINAL PROVENANCE RESULT:
{
  "status": "success",
  "processed_at": "2026-06-30T13:51:00.994813",
  "agent": "provenance",
  "file_path": "/content/canonical_video.mp4",
  "file_hash": "c954f16f0feddbb4d44182f7afd773bdddca29c141bb31b838d2a97bbe3629c6",
  "metadata": {
    "raw": {},
    "flags": [
      "NO_VIDEO_METADATA: No metadata tags found in video/audio file"
    ]
  },
  "c2pa": {
    "c2pa_present": false,
    "c2pa_valid": false,
    "c2pa_issuer": null,
    "c2pa_timestamp": null,
    "c2pa_claims": [],
    "note": "No C2PA signature detected in file"
  },
  "hash_lookup": {
    "hash_match": false,
    "verdict": "UNKNOWN",
    "source": null,
    "detail": "File not found in any known database \u2014 first-time submission"
  },
  "provenance_score": 35,
  "provenance_reasoning": [
    "NO_C2PA: No content credentials found \u2014 cannot verify origin",
    "HASH_UNKNOWN: First-time submission \u2014 no prior record",
    "METADATA_FLAG: NO_

In [5]:
!pip install exifread mutagen

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.7/59.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.7/195.7 kB 10.5 MB/s eta 0:00:00


In [7]:
# ============================================================
# THE TRUE LENS — FORENSICS AGENT (Agent 3)
# ============================================================
# PURPOSE: Deep-learning microscopic analysis of the media.
# Detects visual artifacts, synthetic audio, and lip-sync drift.
#
# INPUT:  JSON dict from the Provenance Agent (Agent 2)
# OUTPUT: Structured JSON with forensic findings and score
#
# INSTALL (run this cell first in Colab/Kaggle):
# !apt install -y ffmpeg
# !pip install opencv-python-headless numpy Pillow scipy librosa
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# ============================================================

# ── WHAT THIS AGENT CHECKS ────────────────────────────────────
# For IMAGES: Pixel noise analysis, ELA (Error Level Analysis),
#             color distribution, frequency domain (FFT)
# For VIDEO:  Frame consistency, blinking analysis, facial artifacts,
#             compression ghost detection
# For AUDIO:  Spectrogram analysis, pitch naturalness, silence patterns
# For VIDEO+AUDIO: Lip-sync drift detection
# ─────────────────────────────────────────────────────────────

import os
import json
import logging
import tempfile
import subprocess
import numpy as np
from datetime import datetime
from pathlib import Path

import cv2                          # OpenCV for image/video analysis
from PIL import Image, ImageChops, ImageEnhance
import scipy.stats as stats         # For statistical analysis

# Audio analysis
try:
    import librosa
    import librosa.display
    LIBROSA_AVAILABLE = True
except ImportError:
    LIBROSA_AVAILABLE = False
    print("WARNING: librosa not installed. Audio analysis will be skipped.")
    print("Run: pip install librosa")

# ── LOGGING SETUP ──
logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s] [FORENSICS] %(message)s",
    datefmt="%H:%M:%S"
)
log = logging.getLogger(__name__)


# ============================================================
# MODULE 1: ERROR LEVEL ANALYSIS (ELA)
# For images — detects regions that were digitally altered
# Real photos have uniform compression quality throughout.
# Edited/AI regions show different error levels (they "glow")
# ============================================================

def run_ela_analysis(image_path: str, quality: int = 90) -> dict:
    """
    Error Level Analysis: Saves image at known quality, compares
    it to original. Manipulated areas show higher error levels.

    Returns:
    - ela_max_value: max pixel difference (higher = more suspicious)
    - ela_mean_value: average difference across the image
    - ela_suspicious_regions: % of image with high error level
    - flag: whether this image looks manipulated
    """
    log.info("Running Error Level Analysis (ELA)...")

    try:
        original = Image.open(image_path).convert("RGB")

        # Save at controlled quality to a temp file
        with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmp:
            tmp_path = tmp.name
        original.save(tmp_path, "JPEG", quality=quality)

        # Reload the compressed version
        compressed = Image.open(tmp_path).convert("RGB")

        # Calculate difference
        ela_image = ImageChops.difference(original, compressed)
        ela_array = np.array(ela_image)

        # Scale up for visibility
        scale = 10
        ela_array_scaled = np.clip(ela_array * scale, 0, 255)

        ela_max = int(ela_array_scaled.max())
        ela_mean = float(ela_array_scaled.mean())

        # Calculate % of pixels with high error level (suspicious regions)
        threshold = 50
        suspicious_pixels = (ela_array_scaled > threshold).sum()
        total_pixels = ela_array_scaled.size
        suspicious_percent = (suspicious_pixels / total_pixels) * 100

        os.unlink(tmp_path)

        # ── Flag logic ──
        # AI-generated images tend to have high mean ELA uniformly
        # Spliced images have localized high-ELA regions
        flags = []
        if ela_mean > 15:
            flags.append(f"HIGH_ELA_MEAN: Average error level is {ela_mean:.1f} — suggests compression history mismatch")
        if suspicious_percent > 20:
            flags.append(f"LARGE_SUSPICIOUS_REGION: {suspicious_percent:.1f}% of image shows elevated error levels")

        log.info(f"ELA complete — max: {ela_max}, mean: {ela_mean:.2f}, suspicious: {suspicious_percent:.1f}%")

        return {
            "ela_max": ela_max,
            "ela_mean": round(ela_mean, 3),
            "suspicious_region_percent": round(suspicious_percent, 2),
            "flags": flags
        }

    except Exception as e:
        log.warning(f"ELA analysis failed: {e}")
        return {"ela_max": None, "ela_mean": None, "suspicious_region_percent": None,
                "flags": [f"ELA_ERROR: {str(e)}"]}


# ============================================================
# MODULE 2: NOISE & FREQUENCY ANALYSIS
# Real camera photos have natural sensor noise.
# AI-generated images have unnaturally smooth noise patterns.
# ============================================================

def analyze_noise_and_frequency(image_path: str) -> dict:
    """
    Analyzes pixel-level noise and frequency domain characteristics.

    Real photos: organic noise from camera sensors (like film grain)
    AI images: noise is too uniform, too smooth, or has grid patterns
    """
    log.info("Running noise and frequency analysis...")
    flags = []

    try:
        img = cv2.imread(image_path)
        if img is None:
            return {"flags": ["IMAGE_READ_ERROR: Could not read image with OpenCV"]}

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float64)

        # ── Noise Analysis ──
        # Apply a slight blur and subtract to isolate noise
        blurred = cv2.GaussianBlur(gray, (5, 5), 0)
        noise = gray - blurred

        noise_std = float(np.std(noise))
        noise_mean = float(np.mean(np.abs(noise)))

        # ── Frequency Domain Analysis (FFT) ──
        # AI-generated images often show grid artifacts in frequency domain
        fft = np.fft.fft2(gray)
        fft_shift = np.fft.fftshift(fft)
        magnitude = np.log(np.abs(fft_shift) + 1)

        # Check for suspicious frequency peaks (grid artifacts)
        center_y, center_x = magnitude.shape[0] // 2, magnitude.shape[1] // 2
        edge_region = np.concatenate([
            magnitude[:10, :].flatten(),
            magnitude[-10:, :].flatten()
        ])
        center_region = magnitude[center_y-20:center_y+20, center_x-20:center_x+20].flatten()

        freq_ratio = float(np.mean(center_region) / (np.mean(edge_region) + 1e-10))

        # ── Flag logic ──
        if noise_std < 1.5:
            flags.append(f"LOW_NOISE_STD: Noise std={noise_std:.2f} — unnaturally smooth, typical of AI generation")
        if noise_std > 30:
            flags.append(f"HIGH_NOISE: Noise std={noise_std:.2f} — heavy noise or compression artifacts")
        if freq_ratio > 100:
            flags.append(f"FFT_ANOMALY: Unusual frequency concentration — possible grid artifact from upscaling or AI generation")

        log.info(f"Noise analysis — std: {noise_std:.2f}, freq_ratio: {freq_ratio:.2f}")

        return {
            "noise_std": round(noise_std, 3),
            "noise_mean": round(noise_mean, 3),
            "fft_freq_ratio": round(freq_ratio, 3),
            "flags": flags
        }

    except Exception as e:
        log.warning(f"Noise/frequency analysis failed: {e}")
        return {"flags": [f"NOISE_ANALYSIS_ERROR: {str(e)}"]}


# ============================================================
# MODULE 3: FACE ARTIFACT DETECTOR
# For images/videos — checks for unnatural face features
# Deepfakes often have: blurry ear/neck borders, asymmetric eyes,
# missing teeth texture, or unnatural skin smoothness
# ============================================================

def detect_face_artifacts(image_path: str) -> dict:
    """
    Uses OpenCV's face detector to find faces and then checks
    for common deepfake artifacts around facial regions.
    """
    log.info("Running face artifact detection...")
    flags = []

    try:
        img = cv2.imread(image_path)
        if img is None:
            return {"faces_found": 0, "flags": ["IMAGE_READ_ERROR"]}

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # Load OpenCV's pre-trained face detector
        face_cascade = cv2.CascadeClassifier(
            cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
        )
        eye_cascade = cv2.CascadeClassifier(
            cv2.data.haarcascades + "haarcascade_eye.xml"
        )

        faces = face_cascade.detectMultiScale(
            gray, scaleFactor=1.1, minNeighbors=5, minSize=(50, 50)
        )

        if len(faces) == 0:
            log.info("No faces detected in image")
            return {"faces_found": 0, "flags": []}

        log.info(f"Found {len(faces)} face(s)")
        face_analyses = []

        for i, (x, y, w, h) in enumerate(faces):
            face_roi_gray = gray[y:y+h, x:x+w]
            face_roi_color = img[y:y+h, x:x+w]

            # ── Check 1: Border sharpness (deepfakes have blurry edges) ──
            # Compare edge sharpness inside face vs border of face
            inner = face_roi_gray[h//4:3*h//4, w//4:3*w//4]
            border_top = face_roi_gray[:h//8, :]
            border_bot = face_roi_gray[7*h//8:, :]

            inner_laplacian = cv2.Laplacian(inner, cv2.CV_64F).var()
            border_laplacian = (
                cv2.Laplacian(border_top, cv2.CV_64F).var() +
                cv2.Laplacian(border_bot, cv2.CV_64F).var()
            ) / 2

            sharpness_ratio = float(inner_laplacian / (border_laplacian + 1e-10))

            # ── Check 2: Skin smoothness (deepfake faces are too smooth) ──
            skin_noise = float(np.std(face_roi_gray.astype(np.float64)))

            # ── Check 3: Eye detection (deepfakes sometimes miss/distort eyes) ──
            eyes = eye_cascade.detectMultiScale(face_roi_gray, scaleFactor=1.1, minNeighbors=3)
            eye_count = len(eyes)

            # ── Flag logic ──
            face_flags = []
            if sharpness_ratio > 15:
                face_flags.append(f"FACE_{i+1}_BORDER_BLUR: Face border is much blurrier than inner face — deepfake artifact")
            if skin_noise < 8:
                face_flags.append(f"FACE_{i+1}_SMOOTH_SKIN: Skin is unnaturally smooth (noise={skin_noise:.1f}) — AI-smoothed")
            if eye_count == 0:
                face_flags.append(f"FACE_{i+1}_NO_EYES: No eyes detected in face region — possible distortion")
            elif eye_count > 3:
                face_flags.append(f"FACE_{i+1}_EYE_ARTIFACT: Unusual eye count ({eye_count}) — possible facial artifact")

            flags.extend(face_flags)
            face_analyses.append({
                "face_index": i + 1,
                "bounding_box": {"x": int(x), "y": int(y), "w": int(w), "h": int(h)},
                "sharpness_ratio": round(sharpness_ratio, 2),
                "skin_smoothness_noise": round(skin_noise, 2),
                "eyes_detected": int(eye_count),
                "face_flags": face_flags
            })

        log.info(f"Face artifact detection complete — {len(flags)} flags found")
        return {"faces_found": len(faces), "face_analyses": face_analyses, "flags": flags}

    except Exception as e:
        log.warning(f"Face artifact detection failed: {e}")
        return {"faces_found": 0, "flags": [f"FACE_DETECTION_ERROR: {str(e)}"]}


# ============================================================
# MODULE 4: VIDEO TEMPORAL CONSISTENCY CHECKER
# Checks frame-by-frame consistency in videos
# Deepfake videos often have: inconsistent lighting between frames,
# sudden face texture changes, or unnatural blinking patterns
# ============================================================

def analyze_video_temporal_consistency(video_path: str, max_frames: int = 60) -> dict:
    """
    Samples frames from the video and checks for temporal inconsistencies.
    Analyzes: brightness changes, face region stability, blinking.
    """
    log.info(f"Analyzing video temporal consistency (sampling {max_frames} frames)...")
    flags = []

    try:
        cap = cv2.VideoCapture(video_path)

        if not cap.isOpened():
            return {"flags": ["VIDEO_OPEN_ERROR: Could not open video file"]}

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        duration = total_frames / fps if fps > 0 else 0

        log.info(f"Video: {total_frames} frames, {fps:.1f} FPS, {duration:.1f}s")

        # Sample frames evenly throughout the video
        sample_interval = max(1, total_frames // max_frames)
        frame_brightnesses = []
        frame_noise_levels = []

        frame_idx = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            if frame_idx % sample_interval == 0:
                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

                # Record brightness
                frame_brightnesses.append(float(np.mean(gray)))

                # Record noise level
                blurred = cv2.GaussianBlur(gray.astype(np.float64), (5, 5), 0)
                noise = float(np.std(gray.astype(np.float64) - blurred))
                frame_noise_levels.append(noise)

            frame_idx += 1

        cap.release()

        if len(frame_brightnesses) < 2:
            return {"flags": ["TOO_FEW_FRAMES: Not enough frames to analyze"]}

        # ── Brightness consistency analysis ──
        brightness_std = float(np.std(frame_brightnesses))
        brightness_changes = np.diff(frame_brightnesses)
        sudden_changes = np.sum(np.abs(brightness_changes) > 30)

        # ── Noise consistency analysis ──
        noise_std = float(np.std(frame_noise_levels))

        # ── Flag logic ──
        if sudden_changes > 5:
            flags.append(f"BRIGHTNESS_JUMPS: {int(sudden_changes)} sudden brightness changes — possible frame splicing")
        if noise_std > 5:
            flags.append(f"INCONSISTENT_NOISE: Frame noise varies significantly (std={noise_std:.1f}) — inconsistent rendering")
        if brightness_std > 40:
            flags.append(f"HIGH_BRIGHTNESS_VARIANCE: Large brightness variance (std={brightness_std:.1f}) — unusual for real video")

        log.info(f"Temporal analysis complete — brightness_std: {brightness_std:.1f}, sudden_changes: {sudden_changes}")

        return {
            "total_frames": total_frames,
            "fps": round(fps, 2),
            "duration_seconds": round(duration, 2),
            "frames_sampled": len(frame_brightnesses),
            "brightness_std": round(brightness_std, 3),
            "sudden_brightness_changes": int(sudden_changes),
            "noise_std": round(noise_std, 3),
            "flags": flags
        }

    except Exception as e:
        log.warning(f"Video temporal analysis failed: {e}")
        return {"flags": [f"VIDEO_ANALYSIS_ERROR: {str(e)}"]}


# ============================================================
# MODULE 5: AUDIO SPECTROGRAM ANALYSIS
# Analyzes audio for synthetic voice signatures
# Real voices have: natural pitch variation, breathing, micro-pauses
# Synthesized voices: too clean, pitch too regular, no room noise
# ============================================================

def analyze_audio_spectrogram(audio_path: str) -> dict:
    """
    Uses librosa to analyze audio spectral characteristics.
    Detects: synthetic flatness, unnatural pitch regularity, clipping.
    """
    if not LIBROSA_AVAILABLE:
        return {"flags": ["LIBROSA_NOT_INSTALLED: Audio analysis skipped"]}

    log.info("Analyzing audio spectrogram...")
    flags = []

    try:
        # Load audio (librosa handles video files too — extracts audio track)
        y, sr = librosa.load(audio_path, sr=None, mono=True, duration=60)

        if len(y) == 0:
            return {"flags": ["EMPTY_AUDIO: No audio content found"]}

        # ── Spectral Features ──
        # Spectral centroid: where the "center of mass" of the spectrum is
        spectral_centroids = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
        centroid_mean = float(np.mean(spectral_centroids))
        centroid_std = float(np.std(spectral_centroids))

        # Spectral rolloff: frequency below which 85% of energy is concentrated
        rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
        rolloff_mean = float(np.mean(rolloff))

        # Zero crossing rate: how often the signal crosses zero (roughness)
        zcr = librosa.feature.zero_crossing_rate(y)[0]
        zcr_mean = float(np.mean(zcr))
        zcr_std = float(np.std(zcr))

        # ── Pitch Analysis (F0 extraction) ──
        # Real voice: pitch varies naturally. TTS: pitch is too regular.
        f0, voiced_flag, voiced_probs = librosa.pyin(
            y, fmin=librosa.note_to_hz("C2"), fmax=librosa.note_to_hz("C7")
        )
        f0_clean = f0[~np.isnan(f0)]

        if len(f0_clean) > 0:
            pitch_std = float(np.std(f0_clean))
            pitch_mean = float(np.mean(f0_clean))
            voiced_ratio = float(np.sum(voiced_flag) / len(voiced_flag))

            # ── Flag logic for pitch ──
            if pitch_std < 10:
                flags.append(f"FLAT_PITCH: Very low pitch variation (std={pitch_std:.1f}Hz) — robotic/synthetic voice")
            if voiced_ratio > 0.95:
                flags.append("NO_SILENCE: Barely any silence detected — unnatural for human speech")
            if voiced_ratio < 0.1:
                flags.append("MOSTLY_SILENCE: Very little voice content detected")
        else:
            pitch_std = None
            pitch_mean = None
            voiced_ratio = None
            flags.append("NO_PITCH_DETECTED: No voiced segments found — may be ambient noise or music only")

        # ── Clipping Detection ──
        clipping_threshold = 0.99
        clipped_samples = np.sum(np.abs(y) > clipping_threshold)
        clipping_ratio = clipped_samples / len(y)
        if clipping_ratio > 0.01:
            flags.append(f"AUDIO_CLIPPING: {clipping_ratio*100:.1f}% of samples clipped — processing artifact")

        # ── Spectral Flatness (synthetic audio is spectrally flat) ──
        flatness = librosa.feature.spectral_flatness(y=y)[0]
        flatness_mean = float(np.mean(flatness))
        if flatness_mean > 0.1:
            flags.append(f"HIGH_SPECTRAL_FLATNESS: Value={flatness_mean:.3f} — audio may be synthesized or heavily processed")

        log.info(f"Audio analysis complete — pitch_std: {pitch_std}, voiced_ratio: {voiced_ratio}")

        return {
            "duration_seconds": round(len(y) / sr, 2),
            "sample_rate": sr,
            "centroid_mean": round(centroid_mean, 2),
            "centroid_std": round(centroid_std, 2),
            "rolloff_mean": round(rolloff_mean, 2),
            "zcr_mean": round(zcr_mean, 4),
            "zcr_std": round(zcr_std, 4),
            "pitch_mean_hz": round(pitch_mean, 2) if pitch_mean else None,
            "pitch_std_hz": round(pitch_std, 2) if pitch_std else None,
            "voiced_ratio": round(voiced_ratio, 3) if voiced_ratio else None,
            "spectral_flatness": round(flatness_mean, 4),
            "clipping_ratio": round(clipping_ratio, 5),
            "flags": flags
        }

    except Exception as e:
        log.warning(f"Audio spectrogram analysis failed: {e}")
        return {"flags": [f"AUDIO_ANALYSIS_ERROR: {str(e)}"]}


# ============================================================
# MODULE 6: LIP-SYNC DRIFT DETECTOR
# Checks if mouth movements match the audio phonemes
# Deepfakes often re-use video with replaced audio — they desync
# ============================================================

def detect_lipsync_drift(video_path: str) -> dict:
    """
    Detects lip-sync drift by analyzing mouth region motion
    against audio energy changes.

    Method: Compare frame-to-frame changes in the mouth region
    against audio amplitude envelope. If they're uncorrelated → drift.

    Note: Professional-grade lip-sync requires models like SyncNet.
    This is a signal-based approximation suitable for screening.
    """
    log.info("Running lip-sync drift detection...")
    flags = []

    if not LIBROSA_AVAILABLE:
        return {"flags": ["LIBROSA_NOT_INSTALLED: Lip-sync analysis skipped"]}

    try:
        # ── Extract audio from video ──
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
            tmp_audio = tmp.name

        subprocess.run([
            "ffmpeg", "-i", video_path,
            "-vn", "-ar", "16000", "-ac", "1",
            "-y", tmp_audio
        ], capture_output=True)

        audio, sr = librosa.load(tmp_audio, sr=16000)
        os.unlink(tmp_audio)

        # ── Audio amplitude envelope ──
        hop_length = 512
        audio_envelope = librosa.feature.rms(y=audio, hop_length=hop_length)[0]

        # ── Extract mouth region motion from video frames ──
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        face_cascade = cv2.CascadeClassifier(
            cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
        )

        mouth_motion = []
        prev_mouth = None
        frame_count = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret or frame_count > 300:  # Analyze first ~10 seconds at 30fps
                break

            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(80, 80))

            if len(faces) > 0:
                x, y, w, h = faces[0]
                # Mouth is in lower 1/3 of face
                mouth_y_start = y + int(h * 0.65)
                mouth_y_end = y + h
                mouth_roi = gray[mouth_y_start:mouth_y_end, x:x+w]

                if prev_mouth is not None and mouth_roi.shape == prev_mouth.shape:
                    motion = float(np.mean(np.abs(
                        mouth_roi.astype(np.float64) - prev_mouth.astype(np.float64)
                    )))
                    mouth_motion.append(motion)
                else:
                    mouth_motion.append(0.0)

                prev_mouth = mouth_roi.copy()
            else:
                mouth_motion.append(0.0)

            frame_count += 1

        cap.release()

        if len(mouth_motion) < 10:
            return {"flags": ["INSUFFICIENT_FRAMES: Not enough frames with faces for lip-sync analysis"]}

        # ── Align audio and video signals ──
        # Resample audio envelope to match video frame count
        audio_resampled = np.interp(
            np.linspace(0, len(audio_envelope) - 1, len(mouth_motion)),
            np.arange(len(audio_envelope)),
            audio_envelope
        )

        # ── Calculate correlation ──
        if np.std(mouth_motion) > 0 and np.std(audio_resampled) > 0:
            correlation = float(np.corrcoef(mouth_motion, audio_resampled)[0, 1])
        else:
            correlation = 0.0

        # ── Flag logic ──
        if correlation < 0.15:
            flags.append(f"LIP_SYNC_DRIFT: Low correlation ({correlation:.2f}) between mouth motion and audio — possible audio replacement")
        elif correlation < 0.30:
            flags.append(f"WEAK_LIP_SYNC: Moderate correlation ({correlation:.2f}) — slight desync detected")

        log.info(f"Lip-sync correlation: {correlation:.3f}")

        return {
            "frames_analyzed": len(mouth_motion),
            "audio_video_correlation": round(correlation, 4),
            "interpretation": (
                "Strong sync" if correlation >= 0.30 else
                "Weak sync — investigate" if correlation >= 0.15 else
                "Poor sync — likely manipulated"
            ),
            "flags": flags
        }

    except Exception as e:
        log.warning(f"Lip-sync detection failed: {e}")
        return {"flags": [f"LIPSYNC_ERROR: {str(e)}"]}


# ============================================================
# MODULE 7: FORENSICS SCORE CALCULATOR
# ============================================================

def calculate_forensics_score(all_flags: list, media_type: str) -> tuple:
    """
    Calculates forensics sub-score from 0-100.

    Scoring:
    - Start at 85 (assumed innocent until evidence says otherwise)
    - Each flag is weighted by severity
    - Critical flags (LIP_SYNC, AI_SOFTWARE) deduct more
    """
    score = 85
    reasoning = []

    critical_keywords = ["LIP_SYNC_DRIFT", "AI_SOFTWARE", "AI_ENCODER", "FLAT_PITCH", "FACE_BORDER_BLUR"]
    moderate_keywords = ["HIGH_ELA", "LARGE_SUSPICIOUS", "NO_EXIF", "BRIGHTNESS_JUMP", "WEAK_LIP_SYNC"]

    for flag in all_flags:
        is_critical = any(k in flag for k in critical_keywords)
        is_moderate = any(k in flag for k in moderate_keywords)

        if is_critical:
            score -= 20
            reasoning.append(f"CRITICAL_FLAG: {flag}")
        elif is_moderate:
            score -= 10
            reasoning.append(f"MODERATE_FLAG: {flag}")
        else:
            score -= 5
            reasoning.append(f"MINOR_FLAG: {flag}")

    score = max(0, min(100, score))
    return score, reasoning


# ============================================================
# MAIN PIPELINE — RUN FORENSICS AGENT
# ============================================================

def run_forensics_agent(provenance_output: dict) -> dict:
    """
    MASTER FUNCTION — runs the full Forensics Agent pipeline.

    Input:  provenance_output = JSON dict from Agent 2
    Output: structured JSON with forensic findings
    """
    log.info("=" * 50)
    log.info("FORENSICS AGENT STARTING")
    log.info("=" * 50)

    if not provenance_output.get("ready_for_next_agent"):
        return {
            "status": "error",
            "error_message": "Previous agent did not complete successfully",
            "ready_for_next_agent": False
        }

    file_path = provenance_output["file_path"]
    media_type = provenance_output.get("media_type", "")

    # Handle case where media_type comes from ingestor via provenance
    if not media_type:
        # Infer from file extension
        ext = Path(file_path).suffix.lower()
        if ext in [".mp4", ".mov", ".avi", ".webm"]:
            media_type = "video"
        elif ext in [".jpg", ".jpeg", ".png", ".webp"]:
            media_type = "image"
        elif ext in [".mp3", ".wav", ".aac"]:
            media_type = "audio"

    try:
        all_flags = []
        findings = {}

        # ── IMAGE ANALYSIS ──
        if media_type == "image":
            log.info("Running IMAGE forensics pipeline...")

            ela = run_ela_analysis(file_path)
            findings["ela"] = ela
            all_flags.extend(ela.get("flags", []))

            noise = analyze_noise_and_frequency(file_path)
            findings["noise_frequency"] = noise
            all_flags.extend(noise.get("flags", []))

            faces = detect_face_artifacts(file_path)
            findings["face_artifacts"] = faces
            all_flags.extend(faces.get("flags", []))

        # ── VIDEO ANALYSIS ──
        elif media_type == "video":
            log.info("Running VIDEO forensics pipeline...")

            temporal = analyze_video_temporal_consistency(file_path)
            findings["temporal_consistency"] = temporal
            all_flags.extend(temporal.get("flags", []))

            audio = analyze_audio_spectrogram(file_path)
            findings["audio_analysis"] = audio
            all_flags.extend(audio.get("flags", []))

            lipsync = detect_lipsync_drift(file_path)
            findings["lipsync"] = lipsync
            all_flags.extend(lipsync.get("flags", []))

        # ── AUDIO ANALYSIS ──
        elif media_type == "audio":
            log.info("Running AUDIO forensics pipeline...")

            audio = analyze_audio_spectrogram(file_path)
            findings["audio_analysis"] = audio
            all_flags.extend(audio.get("flags", []))

        # ── Calculate score ──
        forensics_score, reasoning = calculate_forensics_score(all_flags, media_type)

        result = {
            "status": "success",
            "processed_at": datetime.now().isoformat(),
            "agent": "forensics",
            "file_path": file_path,
            "media_type": media_type,
            "findings": findings,
            "all_flags": all_flags,
            "forensics_score": forensics_score,
            "forensics_reasoning": reasoning,
            "total_flags_found": len(all_flags),
            "ready_for_next_agent": True
        }

        log.info(f"FORENSICS AGENT COMPLETE ✓ — Score: {forensics_score}/100, Flags: {len(all_flags)}")
        return result

    except Exception as e:
        log.error(f"Forensics Agent FAILED: {str(e)}")
        return {
            "status": "error",
            "error_message": str(e),
            "processed_at": datetime.now().isoformat(),
            "ready_for_next_agent": False
        }


# ============================================================
# ── HOW TO CHANGE FOR YOUR OWN TEST ──
#
# 1. Run Agent 2 and get its JSON output
# 2. Paste that JSON output into agent2_output below
# 3. Make sure "file_path" points to your actual canonical file
# 4. Run this cell
# ============================================================

if __name__ == "__main__":

    # ── PASTE AGENT 2 OUTPUT HERE ──
    agent2_output = {
  "status": "success",
  "processed_at": "2026-06-30T13:51:00.994813",
  "agent": "provenance",
  "file_path": "/content/canonical_video.mp4",
  "file_hash": "c954f16f0feddbb4d44182f7afd773bdddca29c141bb31b838d2a97bbe3629c6",
  "metadata": {
    "raw": {},
    "flags": [
      "NO_VIDEO_METADATA: No metadata tags found in video/audio file"
    ]
  },
  "c2pa": {
    "c2pa_present": 'false',
    "c2pa_valid": 'false',
    "c2pa_issuer": 'null',
    "c2pa_timestamp": 'null',
    "c2pa_claims": [],
    "note": "No C2PA signature detected in file"
  },
  "hash_lookup": {
    "hash_match": 'false',
    "verdict": "UNKNOWN",
    "source": "null",
    "detail": "File not found in any known database \u2014 first-time submission"
  },
  "provenance_score": 35,
  "provenance_reasoning": [
    "NO_C2PA: No content credentials found \u2014 cannot verify origin",
    "HASH_UNKNOWN: First-time submission \u2014 no prior record",
    "METADATA_FLAG: NO_VIDEO_METADATA: No metadata tags found in video/audio file"
  ],
  "chain_of_custody_established": 'false',
  "ready_for_next_agent": "true"
}
    print("\n" + "="*60)
    print("  RUNNING FORENSICS AGENT")
    print("="*60)

    result = run_forensics_agent(agent2_output)

    print("\n>>> FINAL FORENSICS RESULT:")
    print(json.dumps(result, indent=2))


  RUNNING FORENSICS AGENT


/tmp/ipykernel_583/3439247769.py:405: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(audio_path, sr=None, mono=True, duration=60)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)



>>> FINAL FORENSICS RESULT:
{
  "status": "success",
  "processed_at": "2026-06-30T13:54:17.351122",
  "agent": "forensics",
  "file_path": "/content/canonical_video.mp4",
  "media_type": "video",
  "findings": {
    "temporal_consistency": {
      "total_frames": 1737,
      "fps": 23.98,
      "duration_seconds": 72.45,
      "frames_sampled": 63,
      "brightness_std": 11.908,
      "sudden_brightness_changes": 2,
      "noise_std": 1.131,
      "flags": []
    },
    "audio_analysis": {
      "duration_seconds": 60.0,
      "sample_rate": 44100,
      "centroid_mean": 2407.96,
      "centroid_std": 1529.99,
      "rolloff_mean": 4760.58,
      "zcr_mean": 0.0532,
      "zcr_std": 0.0613,
      "pitch_mean_hz": 77.22,
      "pitch_std_hz": 9.81,
      "voiced_ratio": 0.639,
      "spectral_flatness": 0.002,
      "clipping_ratio": 0.0,
      "flags": [
        "FLAT_PITCH: Very low pitch variation (std=9.8Hz) \u2014 robotic/synthetic voice"
      ]
    },
    "lipsync": {
      "f

In [8]:
!apt install -y ffmpeg
!pip install opencv-python-headless numpy Pillow scipy librosa
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 65 not upgraded.
Looking in indexes: https://download.pytorch.org/whl/cu118


In [12]:
# ============================================================
# THE TRUE LENS — CROSS-REFERENCE AGENT (Agent 4)
# ============================================================
# PURPOSE: Bridge technical findings with real-world context.
# Extracts claims from media via OCR + transcription, then
# verifies them against trusted sources and fact-check databases.
#
# INPUT:  JSON dicts from ALL previous agents (2 + 3)
# OUTPUT: Final Authenticity Card with Trust Score
#
# INSTALL (run this cell first in Colab/Kaggle):
# !apt install -y tesseract-ocr ffmpeg
# !pip install pytesseract Pillow requests openai-whisper
# !pip install google-api-python-client  # for Google Fact Check API
# ============================================================

# ── WHAT THIS AGENT DOES ──────────────────────────────────────
# 1. OCR: Extracts text visible in the media (headlines, captions)
# 2. Transcription: Converts speech to text using Whisper
# 3. Claim Detection: Identifies verifiable claims in the text
# 4. Fact-Check Search: Queries Google Fact Check Tools API
# 5. Cheapfake Detection: Checks if real footage is used out of context
# 6. Final Synthesis: Combines ALL 4 agents into the Authenticity Card
# ─────────────────────────────────────────────────────────────

import os
import re
import json
import logging
import tempfile
import subprocess
import requests
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
from PIL import Image

# OCR
try:
    import pytesseract
    TESSERACT_AVAILABLE = True
except ImportError:
    TESSERACT_AVAILABLE = False
    print("WARNING: pytesseract not installed. Run: pip install pytesseract")
    print("Also run: apt install -y tesseract-ocr")

# Whisper for transcription
try:
    import whisper
    WHISPER_AVAILABLE = True
except ImportError:
    WHISPER_AVAILABLE = False
    print("WARNING: whisper not installed. Run: pip install openai-whisper")

# ── LOGGING SETUP ──
logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s] [CROSSREF] %(message)s",
    datefmt="%H:%M:%S"
)
log = logging.getLogger(__name__)


# ============================================================
# ── CONFIGURATION ──
# Put your API keys here if you have them.
# The agent works without them — it will skip those checks.
# ============================================================

# Google Fact Check Tools API (free, get at console.cloud.google.com)
# Enables: searching Google's database of fact-checked claims
GOOGLE_FACTCHECK_API_KEY = ""   # ← PASTE YOUR KEY HERE (or leave blank)

# NewsAPI key (free tier at newsapi.org)
# Enables: searching recent news to verify claims
NEWS_API_KEY = ""               # ← PASTE YOUR KEY HERE (or leave blank)


# ============================================================
# MODULE 1: FRAME EXTRACTOR
# Pulls key frames from video for OCR analysis
# ============================================================

def extract_key_frames(video_path: str, num_frames: int = 5) -> list:
    """
    Extracts evenly spaced frames from a video as PIL Images.
    Used to run OCR on any text visible in the video.
    """
    log.info(f"Extracting {num_frames} key frames for OCR...")
    frames = []

    try:
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        interval = max(1, total_frames // num_frames)

        for i in range(num_frames):
            cap.set(cv2.CAP_PROP_POS_FRAMES, i * interval)
            ret, frame = cap.read()
            if ret:
                pil_image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                frames.append(pil_image)

        cap.release()
        log.info(f"Extracted {len(frames)} frames")

    except Exception as e:
        log.warning(f"Frame extraction failed: {e}")

    return frames


# ============================================================
# MODULE 2: OCR TEXT EXTRACTOR
# Reads text that appears visually in the media
# Headlines, captions, chyrons, watermarks, location tags
# ============================================================

def extract_text_via_ocr(file_path: str, media_type: str) -> dict:
    """
    Runs OCR on the media to extract visible text.
    For images: runs directly on the image.
    For videos: runs on key frames and combines results.
    """
    log.info("Running OCR text extraction...")

    if not TESSERACT_AVAILABLE:
        return {
            "ocr_text": "",
            "ocr_confidence": 0,
            "flags": ["TESSERACT_NOT_INSTALLED: OCR skipped — install pytesseract and tesseract-ocr"]
        }

    extracted_texts = []
    flags = []

    try:
        if media_type == "image":
            img = Image.open(file_path)
            # Enhance for better OCR
            img = img.convert("RGB")

            # Pytesseract config: PSM 3 = fully automatic page segmentation
            custom_config = r"--oem 3 --psm 3"
            text = pytesseract.image_to_string(img, config=custom_config)
            extracted_texts.append(text.strip())

        elif media_type == "video":
            frames = extract_key_frames(file_path, num_frames=8)
            for i, frame in enumerate(frames):
                try:
                    text = pytesseract.image_to_string(frame, config=r"--oem 3 --psm 3")
                    if text.strip():
                        extracted_texts.append(text.strip())
                        log.info(f"Frame {i+1} OCR: {text.strip()[:80]}...")
                except Exception as e:
                    log.warning(f"OCR failed for frame {i+1}: {e}")

        # Combine all unique texts
        combined_text = "\n".join(set(extracted_texts))

        # ── Flag: Check for suspicious overlaid text ──
        suspicious_patterns = [
            r"BREAKING",
            r"LEAKED",
            r"EXCLUSIVE",
            r"SHOCKING",
            r"WATCH BEFORE DELETED",
        ]
        for pattern in suspicious_patterns:
            if re.search(pattern, combined_text, re.IGNORECASE):
                flags.append(f"SENSATIONAL_OVERLAY_TEXT: Found '{pattern}' text in media — often used in misleading content")

        log.info(f"OCR complete — extracted {len(combined_text)} characters")

        return {
            "ocr_text": combined_text,
            "ocr_text_length": len(combined_text),
            "frames_processed": len(extracted_texts),
            "flags": flags
        }

    except Exception as e:
        log.warning(f"OCR extraction failed: {e}")
        return {"ocr_text": "", "flags": [f"OCR_ERROR: {str(e)}"]}


# ============================================================
# MODULE 3: AUDIO TRANSCRIPTION
# Converts speech to text using OpenAI Whisper
# Whisper runs locally — no API key needed
# ============================================================

def transcribe_audio(file_path: str, media_type: str) -> dict:
    """
    Transcribes speech from audio or video files using Whisper.
    Whisper runs 100% locally — no API cost.

    Returns transcribed text and detected language.
    """
    log.info("Transcribing audio with Whisper...")

    if not WHISPER_AVAILABLE:
        return {
            "transcript": "",
            "language": None,
            "flags": ["WHISPER_NOT_INSTALLED: Transcription skipped — install openai-whisper"]
        }

    try:
        # Load Whisper model
        # Options: "tiny", "base", "small", "medium", "large"
        # "base" is good balance of speed/accuracy for Colab
        # Change to "small" or "medium" for better accuracy on longer content
        model = whisper.load_model("base")

        log.info("Whisper model loaded — transcribing...")
        result = model.transcribe(file_path, fp16=False)

        transcript = result["text"].strip()
        language = result.get("language", "unknown")

        log.info(f"Transcription complete — language: {language}, length: {len(transcript)} chars")
        log.info(f"Transcript preview: {transcript[:200]}...")

        return {
            "transcript": transcript,
            "language": language,
            "transcript_length": len(transcript),
            "flags": []
        }

    except Exception as e:
        log.warning(f"Transcription failed: {e}")
        return {"transcript": "", "language": None, "flags": [f"TRANSCRIPTION_ERROR: {str(e)}"]}


# ============================================================
# MODULE 4: CLAIM EXTRACTOR
# Identifies verifiable factual claims from the extracted text
# ============================================================

def extract_claims(ocr_text: str, transcript: str) -> list:
    """
    Combines OCR text and transcript, then extracts verifiable claims.

    A "claim" is a statement that can be verified:
    - Names of people + actions ("President X signed...")
    - Locations + events ("Bombing in city Y...")
    - Statistics ("1 million people...")
    - Dates + events ("Yesterday, the earthquake...")

    This uses pattern matching. For production, replace with
    an NLP model (spaCy NER or Claude API) for better extraction.
    """
    log.info("Extracting verifiable claims from text...")

    all_text = f"{ocr_text}\n{transcript}".strip()

    if not all_text:
        log.info("No text available for claim extraction")
        return []

    claims = []

    # ── Pattern 1: Sentences with named entities (basic heuristic) ──
    sentences = re.split(r"[.!?]\s+", all_text)

    for sentence in sentences:
        sentence = sentence.strip()
        if len(sentence) < 20:
            continue

        # Signs of a verifiable claim:
        has_number = bool(re.search(r"\d+", sentence))
        has_proper_noun = bool(re.search(r"\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b", sentence))
        has_action = bool(re.search(
            r"\b(said|announced|confirmed|denied|killed|arrested|signed|launched|attacked|died|won|lost|declared)\b",
            sentence, re.IGNORECASE
        ))

        # Include sentence if it looks like a verifiable claim
        if (has_number or has_action) and has_proper_noun and len(sentence) > 30:
            claims.append(sentence)

    # Deduplicate and limit to top 5 claims (to avoid API rate limits)
    claims = list(dict.fromkeys(claims))[:5]

    log.info(f"Extracted {len(claims)} verifiable claims")
    for i, c in enumerate(claims):
        log.info(f"  Claim {i+1}: {c[:100]}...")

    return claims


# ============================================================
# MODULE 5: FACT-CHECK QUERY ENGINE
# Searches trusted databases to verify extracted claims
# ============================================================

def query_google_factcheck(claim: str, api_key: str) -> dict:
    """
    Queries Google's Fact Check Tools API.
    Returns fact-check results from organizations like
    Snopes, PolitiFact, AFP, Reuters, etc.

    API docs: https://developers.google.com/fact-check/tools/api
    """
    try:
        url = "https://factchecktools.googleapis.com/v1alpha1/claims:search"
        params = {
            "key": api_key,
            "query": claim,
            "languageCode": "en",
            "pageSize": 3
        }

        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()

        if "claims" in data and data["claims"]:
            results = []
            for item in data["claims"][:3]:
                review = item.get("claimReview", [{}])[0]
                results.append({
                    "claim_text": item.get("text", ""),
                    "claimant": item.get("claimant", "Unknown"),
                    "verdict": review.get("textualRating", "Unknown"),
                    "source": review.get("publisher", {}).get("name", "Unknown"),
                    "url": review.get("url", "")
                })
            return {"found": True, "results": results}

        return {"found": False, "results": []}

    except Exception as e:
        log.warning(f"Google Fact Check API failed: {e}")
        return {"found": False, "error": str(e), "results": []}


def query_news_api(claim: str, api_key: str) -> dict:
    """
    Searches recent news headlines for the claim topic.
    Helps identify if an event actually happened.
    """
    try:
        # Extract key terms (first 5 words of claim)
        search_query = " ".join(claim.split()[:6])

        url = "https://newsapi.org/v2/everything"
        params = {
            "apiKey": api_key,
            "q": search_query,
            "sortBy": "relevancy",
            "pageSize": 3,
            "language": "en"
        }

        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()

        articles = data.get("articles", [])
        if articles:
            return {
                "found": True,
                "articles": [
                    {
                        "title": a["title"],
                        "source": a["source"]["name"],
                        "published_at": a["publishedAt"],
                        "url": a["url"]
                    }
                    for a in articles[:3]
                ]
            }

        return {"found": False, "articles": []}

    except Exception as e:
        log.warning(f"News API failed: {e}")
        return {"found": False, "error": str(e), "articles": []}


def verify_claims(claims: list) -> dict:
    """
    Runs each extracted claim through available fact-check APIs.
    Returns aggregated verification results.
    """
    log.info(f"Verifying {len(claims)} claims against fact-check databases...")
    flags = []
    claim_results = []
    debunked_count = 0
    verified_count = 0

    for claim in claims:
        result = {
            "claim": claim,
            "factcheck_result": None,
            "news_result": None,
            "verdict": "UNVERIFIED"
        }

        # ── Google Fact Check ──
        if GOOGLE_FACTCHECK_API_KEY:
            fc_result = query_google_factcheck(claim, GOOGLE_FACTCHECK_API_KEY)
            result["factcheck_result"] = fc_result

            if fc_result["found"]:
                for fc in fc_result["results"]:
                    verdict_lower = str(fc.get("verdict", "")).lower()
                    if any(word in verdict_lower for word in ["false", "misleading", "fake", "pants on fire", "incorrect"]):
                        result["verdict"] = "DEBUNKED"
                        debunked_count += 1
                        flags.append(f"CLAIM_DEBUNKED: '{claim[:60]}...' — rated '{fc['verdict']}' by {fc['source']}")
                        break
                    elif any(word in verdict_lower for word in ["true", "correct", "accurate", "verified"]):
                        result["verdict"] = "VERIFIED"
                        verified_count += 1

        # ── News API ──
        if NEWS_API_KEY:
            news_result = query_news_api(claim, NEWS_API_KEY)
            result["news_result"] = news_result

        claim_results.append(result)
        log.info(f"Claim verdict: {result['verdict']} — '{claim[:60]}...'")

    return {
        "claims_checked": len(claims),
        "verified_count": verified_count,
        "debunked_count": debunked_count,
        "unverified_count": len(claims) - verified_count - debunked_count,
        "claim_results": claim_results,
        "flags": flags
    }


# ============================================================
# MODULE 6: CHEAPFAKE DETECTOR
# Detects real videos used with false context
# ============================================================

def detect_cheapfake(
    ocr_text: str,
    transcript: str,
    forensics_output: dict
) -> dict:
    """
    Detects "cheapfakes" — real, unaltered videos presented
    with false context (wrong date, wrong location, wrong claim).

    Checks:
    1. Date/time claims vs metadata timestamps
    2. Location claims vs metadata GPS (if available)
    3. Low forensics flags but suspicious contextual claims
    """
    log.info("Running cheapfake detection...")
    flags = []
    all_text = f"{ocr_text} {transcript}".lower()

    # ── Check 1: Urgency language patterns common in cheapfakes ──
    cheapfake_phrases = [
        r"happening now",
        r"just happened",
        r"breaking news",
        r"watch before",
        r"they don't want you to see",
        r"share before deleted",
        r"leaked footage",
        r"exclusive footage",
        r"unedited",
        r"real footage"
    ]

    found_phrases = []
    for phrase in cheapfake_phrases:
        if re.search(phrase, all_text, re.IGNORECASE):
            found_phrases.append(phrase)

    if found_phrases:
        flags.append(f"CHEAPFAKE_LANGUAGE: Urgency/manipulation phrases found: {found_phrases}")

    # ── Check 2: Low forensics score but claims seem extraordinary ──
    forensics_score = forensics_output.get("forensics_score", 100)
    total_forensics_flags = forensics_output.get("total_flags_found", 0)

    extraordinary_words = ["explosion", "attack", "killed", "assassination", "disaster", "war", "coup"]
    has_extraordinary_claim = any(word in all_text for word in extraordinary_words)

    if has_extraordinary_claim and total_forensics_flags == 0:
        flags.append("UNVERIFIED_EXTRAORDINARY_CLAIM: Extraordinary event claimed but no forensic manipulation detected — could be real footage with false context")

    # ── Check 3: Year mismatch in text ──
    current_year = datetime.now().year
    year_pattern = re.findall(r"\b(20\d{2})\b", all_text)
    past_years = [int(y) for y in year_pattern if int(y) < current_year - 1]

    if past_years:
        flags.append(f"OLD_DATE_IN_TEXT: Text mentions past year(s) {past_years} — may be old footage reposted as new")

    return {
        "cheapfake_phrases_found": found_phrases,
        "extraordinary_claim_detected": has_extraordinary_claim,
        "old_dates_in_text": past_years,
        "flags": flags
    }


# ============================================================
# MODULE 7: FINAL SYNTHESIS ENGINE
# Combines ALL 4 agents into the Authenticity Card
# ============================================================

def synthesize_final_verdict(
    provenance_output: dict,
    forensics_output: dict,
    crossref_flags: list,
    claim_verification: dict,
    cheapfake_result: dict
) -> dict:
    """
    Produces the final Trust Score and Authenticity Card.

    Weighting:
    - Provenance Score:  25% weight
    - Forensics Score:   40% weight  (most important — technical proof)
    - Cross-Ref Score:   35% weight
    """

    # ── Get sub-scores ──
    provenance_score = provenance_output.get("provenance_score", 50)
    forensics_score = forensics_output.get("forensics_score", 85)

    # ── Calculate Cross-Reference Score ──
    crossref_score = 70  # Start neutral

    if claim_verification["debunked_count"] > 0:
        crossref_score -= claim_verification["debunked_count"] * 25
    if claim_verification["verified_count"] > 0:
        crossref_score += claim_verification["verified_count"] * 15
    if cheapfake_result["flags"]:
        crossref_score -= len(cheapfake_result["flags"]) * 10

    crossref_score = max(0, min(100, crossref_score))

    # ── Weighted Trust Score ──
    trust_score = (
        provenance_score * 0.25 +
        forensics_score  * 0.40 +
        crossref_score   * 0.35
    )
    trust_score = round(max(0, min(100, trust_score)), 1)

    # ── Determine Verdict Label ──
    if trust_score >= 80:
        verdict = "LIKELY AUTHENTIC"
        verdict_color = "GREEN"
    elif trust_score >= 55:
        verdict = "UNCERTAIN — REVIEW NEEDED"
        verdict_color = "YELLOW"
    elif trust_score >= 30:
        verdict = "LIKELY MANIPULATED"
        verdict_color = "ORANGE"
    else:
        verdict = "ALMOST CERTAINLY FAKE"
        verdict_color = "RED"

    # ── Collect all reasoning ──
    all_reasoning = []
    all_reasoning.extend(provenance_output.get("provenance_reasoning", []))
    all_reasoning.extend(forensics_output.get("forensics_reasoning", []))
    all_reasoning.extend([f"CROSSREF: {f}" for f in crossref_flags])
    all_reasoning.extend([f"CHEAPFAKE: {f}" for f in cheapfake_result.get("flags", [])])

    # ── Build Authenticity Card ──
    authenticity_card = {
        "verdict": verdict,
        "verdict_color": verdict_color,
        "trust_score": trust_score,
        "score_breakdown": {
            "provenance_score": provenance_score,
            "forensics_score": forensics_score,
            "crossref_score": crossref_score,
        },
        "weights_used": {
            "provenance": "25%",
            "forensics": "40%",
            "crossref": "35%"
        },
        "key_findings": all_reasoning[:10],  # Top 10 most important findings
        "claims_checked": claim_verification["claims_checked"],
        "claims_debunked": claim_verification["debunked_count"],
        "claims_verified": claim_verification["verified_count"],
        "cheapfake_indicators": cheapfake_result["flags"],
        "chain_of_custody": provenance_output.get("chain_of_custody_established", False),
        "c2pa_verified": provenance_output.get("c2pa", {}).get("c2pa_valid", False),
    }

    return trust_score, verdict, authenticity_card


# ============================================================
# MAIN PIPELINE — RUN CROSS-REFERENCE AGENT
# ============================================================

def run_crossref_agent(provenance_output: dict, forensics_output: dict) -> dict:
    """
    MASTER FUNCTION — runs the Cross-Reference Agent and produces
    the final Authenticity Card.

    Input:  provenance_output = JSON from Agent 2
            forensics_output  = JSON from Agent 3
    Output: Final Authenticity Card JSON
    """
    log.info("=" * 50)
    log.info("CROSS-REFERENCE AGENT STARTING")
    log.info("=" * 50)

    if not forensics_output.get("ready_for_next_agent"):
        return {
            "status": "error",
            "error_message": "Forensics Agent did not complete successfully",
            "authenticity_card": None
        }

    file_path = forensics_output["file_path"]
    media_type = forensics_output["media_type"]

    try:
        all_crossref_flags = []

        # ── STEP 1: OCR extraction ──
        ocr_result = extract_text_via_ocr(file_path, media_type)
        all_crossref_flags.extend(ocr_result.get("flags", []))
        ocr_text = ocr_result.get("ocr_text", "")

        # ── STEP 2: Audio transcription ──
        transcript_result = {"transcript": "", "language": None, "flags": []}
        if media_type in ("video", "audio"):
            transcript_result = transcribe_audio(file_path, media_type)
            all_crossref_flags.extend(transcript_result.get("flags", []))

        transcript = transcript_result.get("transcript", "")

        # ── STEP 3: Extract claims ──
        claims = extract_claims(ocr_text, transcript)

        # ── STEP 4: Verify claims ──
        claim_verification = verify_claims(claims)
        all_crossref_flags.extend(claim_verification.get("flags", []))

        # ── STEP 5: Cheapfake detection ──
        cheapfake_result = detect_cheapfake(ocr_text, transcript, forensics_output)
        all_crossref_flags.extend(cheapfake_result.get("flags", []))

        # ── STEP 6: Final synthesis ──
        trust_score, verdict, authenticity_card = synthesize_final_verdict(
            provenance_output,
            forensics_output,
            all_crossref_flags,
            claim_verification,
            cheapfake_result
        )

        # ── Build final output ──
        result = {
            "status": "success",
            "processed_at": datetime.now().isoformat(),
            "agent": "cross_reference",
            "file_path": file_path,
            "ocr_text": ocr_text,
            "transcript": transcript,
            "language_detected": transcript_result.get("language"),
            "claims_extracted": claims,
            "claim_verification": claim_verification,
            "cheapfake_analysis": cheapfake_result,
            "crossref_flags": all_crossref_flags,
            "authenticity_card": authenticity_card,
            "pipeline_complete": True
        }

        # ── Pretty print the Authenticity Card ──
        log.info("\n" + "="*50)
        log.info("🎯  AUTHENTICITY CARD — FINAL VERDICT")
        log.info("="*50)
        log.info(f"  VERDICT:     {verdict}")
        log.info(f"  TRUST SCORE: {trust_score}/100")
        log.info(f"  Provenance:  {authenticity_card['score_breakdown']['provenance_score']}/100")
        log.info(f"  Forensics:   {authenticity_card['score_breakdown']['forensics_score']}/100")
        log.info(f"  Cross-Ref:   {authenticity_card['score_breakdown']['crossref_score']}/100")
        log.info("="*50)

        return result

    except Exception as e:
        log.error(f"Cross-Reference Agent FAILED: {str(e)}")
        import traceback
        traceback.print_exc()
        return {
            "status": "error",
            "error_message": str(e),
            "processed_at": datetime.now().isoformat(),
            "authenticity_card": None,
            "pipeline_complete": False
        }


# ============================================================
# ── HOW TO CHANGE FOR YOUR OWN TEST ──
#
# 1. Run Agent 2 and Agent 3 and save their JSON outputs
# 2. Paste BOTH outputs below
# 3. Optionally: add your API keys at the top of the file
# 4. Run this cell
# ============================================================

if __name__ == "__main__":

    # ── PASTE AGENT 2 OUTPUT HERE ──
    agent2_output = {
  "status": "success",
  "processed_at": "2026-06-30T13:51:00.994813",
  "agent": "provenance",
  "file_path": "/content/canonical_video.mp4",
  "file_hash": "c954f16f0feddbb4d44182f7afd773bdddca29c141bb31b838d2a97bbe3629c6",
  "metadata": {
    "raw": {},
    "flags": [
      "NO_VIDEO_METADATA: No metadata tags found in video/audio file"
    ]
  },
  "c2pa": {
    "c2pa_present": 'false',
    "c2pa_valid": 'false',
    "c2pa_issuer": 'null',
    "c2pa_timestamp": 'null',
    "c2pa_claims": [],
    "note": "No C2PA signature detected in file"
  },
  "hash_lookup": {
    "hash_match": 'false',
    "verdict": "UNKNOWN",
    "source": "null",
    "detail": "File not found in any known database \u2014 first-time submission"
  },
  "provenance_score": 35,
  "provenance_reasoning": [
    "NO_C2PA: No content credentials found \u2014 cannot verify origin",
    "HASH_UNKNOWN: First-time submission \u2014 no prior record",
    "METADATA_FLAG: NO_VIDEO_METADATA: No metadata tags found in video/audio file"
  ],
  "chain_of_custody_established": 'false',
  "ready_for_next_agent": "true"
}

    # ── PASTE AGENT 3 OUTPUT HERE ──
    agent3_output = {
  "status": "success",
  "processed_at": "2026-06-30T13:54:17.351122",
  "agent": "forensics",
  "file_path": "/content/canonical_video.mp4",
  "media_type": "video",
  "findings": {
    "temporal_consistency": {
      "total_frames": 1737,
      "fps": 23.98,
      "duration_seconds": 72.45,
      "frames_sampled": 63,
      "brightness_std": 11.908,
      "sudden_brightness_changes": 2,
      "noise_std": 1.131,
      "flags": []
    },
    "audio_analysis": {
      "duration_seconds": 60.0,
      "sample_rate": 44100,
      "centroid_mean": 2407.96,
      "centroid_std": 1529.99,
      "rolloff_mean": 4760.58,
      "zcr_mean": 0.0532,
      "zcr_std": 0.0613,
      "pitch_mean_hz": 77.22,
      "pitch_std_hz": 9.81,
      "voiced_ratio": 0.639,
      "spectral_flatness": 0.002,
      "clipping_ratio": 0.0,
      "flags": [
        "FLAT_PITCH: Very low pitch variation (std=9.8Hz) \u2014 robotic/synthetic voice"
      ]
    },
    "lipsync": {
      "frames_analyzed": 301,
      "audio_video_correlation": -0.0008,
      "interpretation": "Poor sync \u2014 likely manipulated",
      "flags": [
        "LIP_SYNC_DRIFT: Low correlation (-0.00) between mouth motion and audio \u2014 possible audio replacement"
      ]
    }
  },
  "all_flags": [
    "FLAT_PITCH: Very low pitch variation (std=9.8Hz) \u2014 robotic/synthetic voice",
    "LIP_SYNC_DRIFT: Low correlation (-0.00) between mouth motion and audio \u2014 possible audio replacement"
  ],
  "forensics_score": 45,
  "forensics_reasoning": [
    "CRITICAL_FLAG: FLAT_PITCH: Very low pitch variation (std=9.8Hz) \u2014 robotic/synthetic voice",
    "CRITICAL_FLAG: LIP_SYNC_DRIFT: Low correlation (-0.00) between mouth motion and audio \u2014 possible audio replacement"
  ],
  "total_flags_found": 2,
  "ready_for_next_agent": "true"
}

    print("\n" + "="*60)
    print("  RUNNING CROSS-REFERENCE AGENT (Final Agent)")
    print("="*60)

    result = run_crossref_agent(agent2_output, agent3_output)

    print("\n>>> FINAL AUTHENTICITY CARD:")
    print(json.dumps(result.get("authenticity_card"), indent=2))

    print("\n>>> FULL PIPELINE OUTPUT:")
    print(json.dumps(result, indent=2))


  RUNNING CROSS-REFERENCE AGENT (Final Agent)


100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 112MiB/s]



>>> FINAL AUTHENTICITY CARD:
{
  "verdict": "LIKELY MANIPULATED",
  "verdict_color": "ORANGE",
  "trust_score": 51.2,
  "score_breakdown": {
    "provenance_score": 35,
    "forensics_score": 45,
    "crossref_score": 70
  },
  "weights_used": {
    "provenance": "25%",
    "forensics": "40%",
    "crossref": "35%"
  },
  "key_findings": [
    "NO_C2PA: No content credentials found \u2014 cannot verify origin",
    "HASH_UNKNOWN: First-time submission \u2014 no prior record",
    "METADATA_FLAG: NO_VIDEO_METADATA: No metadata tags found in video/audio file",
    "CRITICAL_FLAG: FLAT_PITCH: Very low pitch variation (std=9.8Hz) \u2014 robotic/synthetic voice",
    "CRITICAL_FLAG: LIP_SYNC_DRIFT: Low correlation (-0.00) between mouth motion and audio \u2014 possible audio replacement"
  ],
  "claims_checked": 0,
  "claims_debunked": 0,
  "claims_verified": 0,
  "cheapfake_indicators": [],
  "chain_of_custody": "false",
  "c2pa_verified": "false"
}

>>> FULL PIPELINE OUTPUT:
{
  "status":

In [10]:
!apt install -y tesseract-ocr ffmpeg
!pip install pytesseract Pillow requests openai-whisper
!pip install google-api-python-client  # for Google Fact Check API

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 65 not upgraded.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 13.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 5.6 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=2931fcaef3deb39313e8f9790f0947fb333ccc657a93dd7e7c840853d5bcc2e6
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper
